# Playfit Intelligence Lab — 04: Model Evaluation

Evaluación offline del recomendador híbrido con métricas de precisión, cobertura, novedad y diversidad.

In [ ]:
import sys; sys.path.insert(0, '..')
import warnings; warnings.filterwarnings('ignore')

import numpy as np
import polars as pl
import matplotlib.pyplot as plt
import seaborn as sns

from src.features.game_features import build_feature_matrix, compute_popularity_score, compute_richness_score
from src.models.content_based import build_content_model
from src.models.hybrid import HybridRecommender
from src.evaluation.metrics import (
    precision_at_k, recall_at_k, ndcg_at_k, map_at_k,
    hit_rate_at_k, coverage, novelty, diversity,
)

sns.set_theme(style="darkgrid")
plt.rcParams['figure.figsize'] = (12, 6)
print("Librerías y módulos cargados.")

## 1. Setup del Modelo y Datos de Evaluación

In [ ]:
fm = build_feature_matrix()
fm = compute_popularity_score(fm)
fm = compute_richness_score(fm)
cm = build_content_model(fm, n_components=100)

rec = HybridRecommender(alpha=0.5, beta=0.4, gamma=0.1)
rec.fit(fm, cm)

popularity_map = dict(zip(fm['game_id'].to_list(), fm['popularity_score'].to_list()))

# Perfiles de usuario sintéticos para evaluación
test_profiles = [
    {"name": "Casual Gamer", "tags": ["casual", "puzzle", "family", "party", "rhythm"]},
    {"name": "Hardcore Gamer", "tags": ["action", "challenging", "souls_like", "rpg", "shooter"]},
    {"name": "Story Seeker", "tags": ["story_rich", "narrative", "adventure", "atmospheric", "single_player"]},
    {"name": "Strategy Fan", "tags": ["strategy", "tactical", "simulation", "management", "turn_based"]},
    {"name": "Retro Player", "tags": ["retro", "arcade", "platformer", "pixel_art", "2d_flat"]},
]

all_recommendations = []
all_relevant = []
for profile in test_profiles:
    recs = rec.recommend_for_profile(profile['tags'], k=20)
    all_recommendations.append([r['game_id'] for r in recs])
    relevant_tags = set(profile['tags'])
    relevant = set()
    for i, row in enumerate(fm.to_dicts()):
        if any(tag in relevant_tags for tag in profile['tags'] if tag in row and row[tag] == 1):
            relevant.add(row['game_id'])
    all_relevant.append(relevant)

print(f"{'Perfil':20s} {'Relevantes':>12s} {'Recomendados':>14s}")
print("-" * 50)
for p, recs, rel in zip(test_profiles, all_recommendations, all_relevant):
    print(f"{p['name']:20s} {len(rel):>12,} {len(recs):>14,}")

## 2. Métricas Core: Precision@k, Recall@k, NDCG@k

In [ ]:
ks = [1, 3, 5, 10, 20]
metrics_by_k = {}

for k in ks:
    precs = [precision_at_k(rec, rel, k) for rec, rel in zip(all_recommendations, all_relevant)]
    recs_m = [recall_at_k(rec, rel, k) for rec, rel in zip(all_recommendations, all_relevant)]
    ndcgs = [ndcg_at_k(rec, rel, k) for rec, rel in zip(all_recommendations, all_relevant)]
    metrics_by_k[k] = {
        'precision': np.mean(precs),
        'recall': np.mean(recs_m),
        'ndcg': np.mean(ndcgs),
    }

print(f"{'k':>5} {'Precision@k':>14} {'Recall@k':>12} {'NDCG@k':>10}")
print("-" * 45)
for k in ks:
    m = metrics_by_k[k]
    print(f"{k:>5} {m['precision']:>14.4f} {m['recall']:>12.4f} {m['ndcg']:>10.4f}")

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
for metric in ['precision', 'recall', 'ndcg']:
    values = [metrics_by_k[k][metric] for k in ks]
    ax.plot(ks, values, marker='o', label=metric)
ax.set_xlabel('k')
ax.set_ylabel('Score')
ax.set_title('Métricas de Evaluación vs k')
ax.legend()
ax.grid(True, alpha=0.3)
plt.savefig('../reports/figures/metrics_vs_k.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. Cobertura del Catálogo

In [ ]:
total_catalog = len(fm)
for profile, recs in zip(test_profiles, all_recommendations):
    cov = coverage([recs], total_catalog)
    print(f"{profile['name']:20s} Coverage: {cov*100:.2f}% ({int(cov*total_catalog):,} de {total_catalog:,} juegos)")

combined_coverage = coverage(all_recommendations, total_catalog)
print(f"\n{'Combinado':20s} Coverage: {combined_coverage*100:.2f}% ({int(combined_coverage*total_catalog):,} de {total_catalog:,} juegos)")

## 4. Novedad (Novelty)

In [ ]:
for profile, recs in zip(test_profiles, all_recommendations):
    nov = novelty([recs], popularity_map)
    print(f"{profile['name']:20s} Novelty: {nov:.4f}")

## 5. Diversidad Intra-lista

In [ ]:
for profile, recs in zip(test_profiles, all_recommendations):
    div = diversity([recs], rec, k=5)
    print(f"{profile['name']:20s} Diversity@5: {div:.4f}")

## 6. Cold-Start Analysis

In [ ]:
confidence_segments = [(0, 30, 'Baja'), (30, 60, 'Media'), (60, 80, 'Alta'), (80, 100, 'Muy alta')]
print(f"{'Segmento':15s} {'Juegos':>10s} {'En Top-20':>10s} {'Proporción':>10s}")
print("-" * 47)
all_top20 = set()
for recs in all_recommendations:
    all_top20.update(recs[:20])

for lo, hi, label in confidence_segments:
    mask = (fm['data_confidence_score'] >= lo) & (fm['data_confidence_score'] < hi)
    count = mask.sum()
    in_top = sum(fm.filter(mask)['game_id'].is_in(list(all_top20)))
    prop = in_top / count if count > 0 else 0
    print(f"{label:15s} {count:>10,} {in_top:>10,} {prop:>10.3f}")

## 7. Resumen de Evaluación

In [ ]:
print("=" * 60)
print("RESUMEN DE EVALUACIÓN - PLAYFIT RECOMMENDER")
print("=" * 60)
m5 = metrics_by_k[5]
print(f"Precision@5 (promedio):    {m5['precision']:.4f}")
print(f"Recall@5 (promedio):       {m5['recall']:.4f}")
print(f"NDCG@5 (promedio):         {m5['ndcg']:.4f}")
print(f"MAP@20 (promedio):         {map_at_k(all_recommendations, all_relevant, 20):.4f}")
print(f"Hit Rate@20:               {hit_rate_at_k(all_recommendations, all_relevant, 20):.4f}")
print(f"Cobertura del catálogo:    {combined_coverage*100:.2f}%")
print()
print("Mejoras potenciales:")
print("  - Incorporar feedback implícito (tiempo de juego, clicks)")
print("  - Añadir filtrado colaborativo (matrix factorization)")
print("  - Optimizar α/β/γ con validación cruzada sobre datos de interacción reales")
print("  - Añadir re-ranking de diversidad (MMR)")
print("=" * 60)